In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-8: Where do the weights come from?

## RL for LLMs, Part 2 (RFT alternative methods), session 8 of 14

## What this session teaches

PPO, GRPO, SFT and distillation look like four different methods, but they all do the same thing.

At each token the model's parameters are nudged. The nudge is a number, the weight, times the direction that makes that token more likely.

A positive weight makes the token more likely, a negative weight makes it less likely, and zero leaves it alone.

The methods differ in two ways only.

First, where the weight comes from. It can come from something the model should copy, such as the correct token in a dataset or a teacher's probabilities, or from a reward it should earn, such as a verifier's score.

Second, how fine the weights are: one for a whole answer, one per token, or one for every possible next token.

The deck then shows one consequence. If a model learns from its own answers and every weight is equal, nothing changes on average. Learning from your own output needs a signal from outside.

The point of the deck is that you can look at any training method and ask the same two questions about it: where do its weights come from, and how fine are they?

By the end you can

- say what a weight is, and that every method is "weights times that direction"
- say where a method's weights come from and how fine they are
- say why training on your own answers with equal weights goes nowhere

**Where this session fits: the topics of the series, in order**

<img src="images/rllm8v2/series_strip_7.png" width=88%>

- this session starts topic 7: who is the teacher, where the weights come from


## Setting and words

**Why this slide?** The first sections use these words; each is said by what it does.

- An **update** changes the model's **parameters** (its learned numbers)
- A **weight** multiplies the gradient of a token's log probability in an update
  - a weight is not a parameter
- A **push** is the change in that token's probability: up for a positive weight, down for a negative one
- A **prefix** is the prompt plus the completion so far
- A **candidate token** is any token the model could write at a position
- A **reference** is what an update copies: a dataset's token, or a **teacher**'s probabilities (a frozen model)
  - the model being trained is the **student**
- A **reward** is what a verifier (a program, RLLM-1) or a reward model (a trained network, RLLM-2) gives a completion
  - a **baseline** is a comparison level subtracted from it


## The map: six parts

**Why this slide?** One form of update, two questions about it. These are the parts, in the order we take them.

<img src="images/rllm8v2/deck_map_all.png" width=70% alt="Deck map: the axes, Form, No-weights case, where the weights come from, Granularity and Placements in a chain.">

- RLLM-7 settled whose samples an update uses
- this session turns to what an update does with them
- **The axes:** the three questions, and the two this session teaches
- **Form:** the one sum every method shares
- **No-weights case:** why the weights must come from outside
- **Where the weights come from, and how fine they are**, then **placements** of four methods


## Where the weights come from: the picture

**Why this slide?** Every method puts its weights into the same sum. The only question is where each weight comes from.

<img src="images/rllm8v2/weight_source.png" width=50% alt="A reference (dataset token or teacher probabilities) or a reward (from a verifier or reward model, minus a baseline) supplies the weights, which enter the same sum.">

- a **reference** gives weights to copy: a dataset's token, or a teacher's probabilities
- a **reward** gives weights to earn, after a baseline is subtracted
- how fine the weights are is the second question, later


## The axes: three questions, and the two this session teaches

*You are here: The axes.*

<img src="images/rllm8v2/deck_map_axes.png" width=54% alt="Deck map with the axes section highlighted.">

<table>
<tr><th>Axis</th><th>The question</th><th>The answers</th></tr>
<tr><td>1. Whose samples (RLLM-7)</td><td>whose completions does the update use?</td><td>the model's own, or another model's or a dataset's</td></tr>
<tr style="background:#fde3c4"><td><b>2. Where the weights come from</b></td><td><b>something to imitate, or to maximize?</b></td><td><b>a reference to imitate, or a reward to maximize</b></td></tr>
<tr style="background:#fde3c4"><td><b>3. How fine</b></td><td><b>how many weights does a sample carry?</b></td><td><b>one per completion, one per sampled token, or a distribution per token</b></td></tr>
</table>

<span style="color:#e8820c">&#9632;</span> <i>orange rows: the axes this session teaches</i>

## The axes: where RLLM-7's prefix choice went

> **RLLM-7's columns were which prefix. Today's columns are where the weights come from; the rows (whose samples) are unchanged.**

<img src="images/rllm8v2/rllm7_grid.png" width=46% alt="RLLM-7's grid: rows others samples and own samples, columns imposed prefix and own prefix; cells SFT-like, empty, degenerate, and PPO GRPO on-policy distillation.">

- The prefix choice is folded into the form: the prefix $y_{<t}$ sits in every term of the sum (next section)
- Cell labels (our redrawing):
  - **SFT-like**: others' samples, imposed prefix. **Empty**: others' samples, own prefix
  - **Degenerate**: own samples, imposed prefix
  - **PPO, GRPO** and on-policy distillation (RLLM-11): own samples, own prefix
- Own samples with equal weights is the same situation as the degenerate cell: today's No-weights case

## Form: all methods share one sum

*You are here: Form.*

<img src="images/rllm8v2/deck_map_form.png" width=54% alt="Deck map with the Form section highlighted.">

> **Today's claim (our interpretation, after the DeepSeekMath unified view): each method's update is one sum, a weight times the gradient of a log probability. Methods differ in where the weights come from (decides their values) and how fine they are.**

- A teacher gives a probability for every candidate token, so a weight can sit on tokens the student never wrote

$$\sum_{t}\sum_{v} \; w_t(v)\;\nabla_\theta \log \pi_\theta(v \mid y_{<t})$$

- Next: the weights at one position, then the terms of the sum

## Form: the weights at one position

> **At one position every candidate token has a weight, the multiplier on that token's gradient. Three methods, three patterns of weights.**

<img src="images/rllm8v2/weight_patterns.png" width=54% alt="Bar charts of the weights of 8 candidate tokens at one position: SFT has weight 1 for one token, PPO and GRPO have signed weights for sampled tokens, distillation has a weight for every token.">

- **SFT** (supervised fine-tuning, trains on a dataset): weight 1 on the target's token, 0 elsewhere
  - the target is the one token that incurs no loss: a one-hot, as in cross-entropy
- **PPO, GRPO**: signed weights on the sampled tokens
- **Classical distillation** (RLLM-1: training on a teacher's saved outputs): the teacher's probability on every candidate, our interpretation of saving probabilities, not sampled text
- Distillation is why the sum covers every candidate: the teacher weights tokens the student did not write

## Form: the terms of the sum

$$\sum_{t}\sum_{v} \; w_t(v)\;\nabla_\theta \log \pi_\theta(v \mid y_{<t})$$

> **The sum has one term for every candidate token $v$ at every position, not only for the sampled one.**

- $\pi_\theta$ is the student, $\theta$ the parameters the update changes. The sum is the direction added to $\theta$
- The sum runs over a batch's completions and positions $t$
- $y_{<t}$ is the prefix, $v$ a candidate token, $w_t(v)$ its weight
- $\nabla_\theta\log\pi_\theta(v\mid y_{<t})$ is the **score** of token $v$ (Recall from B1: the gradient of a token's log probability)
- A **common form**, not a common objective: each method has its own loss

## Form: a push is what a weight does

> **A positive weight pushes a token's probability up, a negative one pushes it down. The rest of the session asks where the weights come from, and how fine they are.**

<img src="images/rllm8v2/push_defined.png" width=60% alt="Weights +1, 0 and -1 on three tokens, and the probabilities of those tokens before and after the update.">

- A weight of 0 leaves a token alone, apart from the small change the others cause: probabilities sum to 1
- Next: a case where the weights are all equal

## No-weights case: equal weights on the model's own samples

*You are here: No-weights case.*

<img src="images/rllm8v2/deck_map_nowt.png" width=54% alt="Deck map with the No-weights case section highlighted.">

> **Why must the weights come from outside? Try the simplest case: the model trains on its own samples with equal weights (a thought experiment with an exact answer in expectation, not a method).**

**Equal weights**: every sampled token gets weight 1. That is SFT on the model's own samples.

**Predict before you advance the slide.** The completions are drawn at temperature 1 (Recall from C1: the model's own probabilities). Averaged over many batches, after one update the model is:

- **better**: it trained on its own (reasonable) completions
- **worse**: it trained on its own mistakes as well as its successes
- **unchanged, on average**

## No-weights case: no push on average, until a reward breaks it

**Why this slide?** The last slide asked for a guess. Here is the exact answer, in words and in a small example.

> **RLLM-7 kept this for today: with equal weights, the model's own completions give no push on average (left). A reward gives a push with a direction (right).**

<img src="images/rllm8v2/zero_mean_gradient.png" width=46% alt="Same toy policy twice: with equal weights the average push on each logit is zero; with weights equal to a noisy reward minus a baseline it points toward the better action.">

- **In words:** the model's own samples are drawn in proportion to its probabilities, so each token is reinforced in proportion to its probability
  - the probabilities stay where they are
- **The toy:** 4 actions stand for the candidate tokens at one position
  - its parameters are its logits (Recall from C1: raw outputs before the softmax)
- **In symbols** (Intro B1's zero-mean score lemma):
  - sampling action $a$ pushes its logit up by $1-\pi_a$, each other logit $j$ down by $\pi_j$
  - the average is $\sum_a\pi_a(e_a-\pi)=0$
- Logit 2 is best: pushed up in **82%** of batches with a reward, **45%** with equal weights

## No-weights case: what breaks the zero, in practice

**Why this slide?** If equal weights on its own text do nothing, the weights must come from outside. This slide says what is left to settle.

> **Keep this: equal weights on your own completions give no push on average. It returns in RLLM-9 as the reason for a filter (keep only the completions a reward approves).**

- So the weights must come from outside: a dataset, a teacher, a verifier, a reward model. Which? That is axis 2
- In practice the zero is not exact:
  - a finite batch is noisy (the dots)
  - Adam (the optimizer that rescales each step) can sharpen the model
  - temperature below 1 or top-p (sampling only from the most likely tokens whose probabilities add to $p$) sharpens it
  - GRPO's original Averaging rule divides a completion's weights by its length (RLLM-6), so they are no longer equal. Our interpretation: that breaks the zero
- "Unchanged" means no systematic direction, not no movement at all

<!-- seed-plant: zero-push-equal-weights -->

## Where the weights come from: a reference to copy, or a reward to maximize

*You are here: Where the weights come from (axis 2).*

<img src="images/rllm8v2/deck_map_source.png" width=54% alt="Deck map with the where-the-weights-come-from section highlighted.">

> **Axis 2 asks where the weights come from: a reference (a dataset's token, or a teacher's probabilities) or a reward (from a verifier or a reward model).**

<img src="images/rllm8v2/weight_source.png" width=34% alt="A reference (dataset token or teacher probabilities) or a reward (from a verifier or reward model, minus a baseline) supplies the weights, which enter the same sum.">

- **Reference**: SFT's weight is 1 for the dataset's one token. Classical distillation puts the teacher's probabilities on every candidate
- **Reward**: becomes signed weights on the sampled tokens, after subtracting a baseline
- **Trained or frozen**: the student and PPO's critic are trained
  - the teacher, reward model and verifier stay fixed
- Not RLLM-5's reference model (the frozen KL anchor)

## Where the weights come from: what you have picks the column

> **What you have decides the column: demonstrations give a reference, a verifier gives a reward.**

- A dataset's token raises that token and lowers the others by normalization
  - our interpretation: it cannot mark a sampled sequence as bad
  - a teacher judging the student's own samples can (RLLM-11)
- A reward needs the model to succeed sometimes, for a sparse reward (Recall from RLLM-1: zero for most completions)
- PPO, GRPO: a reward, usually of the whole completion, becomes signed weights after a baseline

## Granularity: how fine are the weights?

*You are here: Granularity (axis 3).*

<img src="images/rllm8v2/deck_map_gran.png" width=54% alt="Deck map with the Granularity section highlighted.">

> **Axis 3 asks how many numbers the weights carry: one per completion, one per sampled token, or a distribution per token.**

<img src="images/rllm8v2/granularity_strip.png" width=34% alt="Three rows over the tokens of one completion: one weight per completion, one weight per sampled token, a distribution of weights per token.">

- One per completion: the episode reward (Recall from C1: one reward for the whole completion of $L$ tokens)
  - **credit assignment** is hard (Recall from A0: deciding which action earned a later reward): which of the $L$ tokens earned it?
- One per sampled token: a weight for each token the model wrote
- A distribution per token: a weight for every candidate token. SFT is the one-hot case

## Granularity: not the same as where the weights come from

> **Granularity is not where the weights come from (axis 2): the same granularity can come from different places.**

<table>
<tr><th>Granularity</th><th>Example method</th><th>Where the weights come from</th></tr>
<tr><td>one per completion</td><td>GRPO (RLLM-6)</td><td>the completion's reward minus the group mean: the same advantage on every token</td></tr>
<tr><td>one per sampled token</td><td>PPO (RLLM-5)</td><td>a critic's advantage estimate (GAE, Recall from B3: blends returns of several depths); a process reward model (RLLM-4: a reward for each step; <i>our interpretation</i>: it is the weight for the step's tokens); or a teacher's per-token signal on the student's samples (RLLM-11)</td></tr>
<tr><td>a distribution per token</td><td>classical distillation; SFT is one-hot</td><td>a reference: the teacher's probabilities, or the dataset's token</td></tr>
</table>

- One weight per completion is the coarse end: GRPO pays for it with credit assignment, as RLLM-5 noted

## Placements: a first map, and how to read it

*You are here: Placements.*

<img src="images/rllm8v2/deck_map_place.png" width=40% alt="Deck map with the Placements section highlighted.">

<img src="images/rllm8v2/first_map.png" width=32% alt="Grid: rows are others samples and own samples, columns are a reference or a reward supplying the weights; SFT, classical distillation, PPO and GRPO placed, and dashed slots for later sessions.">

> **SFT, classical distillation, PPO and GRPO are placed; three dashed slots wait for later sessions.**

- **Rows**: whose samples. **Columns**: where the weights come from. **Axis 3** is inside each box
- **Dashed slots**, filled later by the same axes:
  - **slot 1, RLLM-9** (own samples, both columns): a hybrid
    - a reward picks the completions; those kept are the reference the model trains on
  - **slot 2, RLLM-10** (others' samples, reward column): a preference pair (RLLM-2: a better and a worse completion of one prompt)
    - our interpretation: RLLM-1 to 6 reward the model's own completion, so this placement is ours
  - **slot 3, RLLM-11** (own samples, reference column): the teacher weights every candidate at the student's own prefix
    - not SFT on the student's own tokens

## Think about

1. You have demonstrations and a verifier for one task. Which axis-2 column is each in?
   - what must be true of the model for the verifier's weights to be usable (RLLM-1)?
2. Replace the reward function with a reward *model*. Does the answer on any of the three axes change? Argue each.
3. In the RLLM-11 slot, *where* would the reference come from? What would you need in hand?
4. All the completions in a batch earn the same reward. After subtracting the average, what are the weights? What happens to the update?

## What we established, and what it sets up

- **Established**
  - one gradient form; methods differ in the weights
  - own completions with equal weights push nowhere on average
  - **axis 2**: a reference or a reward. **Axis 3**: per completion (credit assignment is hard), per sampled token, or a distribution per token
  - four methods placed on the grid
- **Limit:** a placement says where the weights come from, not whether it works
- **Sets up the three dashed slots**

| Session | Where it meets this |
|---|---|
| **RLLM-9** sample, filter, imitate | own samples, both columns: a reward picks the data |
| **RLLM-10** DPO | others' samples, a reward, from preference pairs |
| **RLLM-11** on-policy distillation | own samples, a reference from a teacher |

## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Shao et al. (2024), *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*](https://arxiv.org/abs/2402.03300): abstract, sections 4.1 and 5.2.1
  - [Nathan Lambert, *Reinforcement Learning (Motivation & Math)*, lecture slides, rlhfbook.com course](https://rlhfbook.com/teach/course/lec3-chap6-p1/): the policy-gradient derivation, REINFORCE, PPO and GRPO in one slide deck
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved

## Aside, justification: why the average push is zero

*For your own review; not worked through in session.*

- The average push is $\mathbb{E}_{y\sim\pi_\theta}\big[\nabla_\theta\log\pi_\theta(y)\big]=\sum_y \pi_\theta(y)\,\nabla_\theta\log\pi_\theta(y)$
- **Log-derivative trick** (Intro B1, RLLM-5): $\nabla\log\pi=\nabla\pi/\pi$, so the $\pi$'s cancel: $\sum_y\nabla_\theta\pi_\theta(y)=\nabla_\theta\sum_y\pi_\theta(y)=\nabla_\theta 1=0$
  - the identity behind the policy gradient, read the other way
- **Toy check** (softmax over 4 actions, parameters = logits $z$): $\nabla_z\log\pi(a)=e_a-\pi$, with $e_a$ one-hot on the sampled action
  - averaging: $\sum_a\pi(a)(e_a-\pi)=\pi-\pi\cdot 1=0$
- **With non-constant weights** $w(a)$, $\sum_a\pi(a)\,w(a)\,(e_a-\pi)\neq 0$ in general: that is the **policy gradient**
- A **constant** weight factors out and gives zero again: subtracting a baseline adds no bias
- All of this holds per token as well as per completion